## Model Training
#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

#### Import the CSV Data as Pandas DataFrame

In [2]:
df = pd.read_csv('data/stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
df['average'] = (df['math_score'] + df['reading_score'] + df['writing_score']) / 3
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,average
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667
1,female,group C,some college,standard,completed,69,90,88,82.333333
2,female,group B,master's degree,standard,none,90,95,93,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333
4,male,group C,some college,standard,none,76,78,75,76.333333


In [4]:
X = df.drop(columns=['average', 'math_score'])
X.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [5]:
y = df['math_score']
print(y)

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64


#### 1.2 Encoding and column transformer
1. we will use one hot encoding(nominal encoding) for following features:

- Gender

- Race/Ethnicity

- Lunch

2. We will use ordinal encoding for follwing features:

- parental education(parental_level_of_education)

- Test preparation course(test_preparation_course)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

ohe_columns=['gender', 'race_ethnicity', 'lunch']
ordinal_columns=['parental_level_of_education', 'test_preparation_course']
scale_columns = ['reading_score', 'writing_score']

education_rank = [
    'some high school', 
    'high school', 
    'some college', 
    'associate\'s degree', 
    'bachelor\'s degree', 
    'master\'s degree'
]
prep_rank = ['none', 'completed']

ohe_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder(categories=[education_rank, prep_rank])
scaler = StandardScaler()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", ohe_transformer, ohe_columns),
        ("OrdinalEncoder", ordinal_transformer, ordinal_columns),
        ("StandardScaler", scaler, scale_columns)
    ]
)

In [7]:
# Tip: Train test split should be done before encoding so that there is no chance for data leakage
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Encoding train and test splits
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [9]:
X_train.shape, X_test.shape

((800, 13), (200, 13))

In [10]:
# Now lets write an evaluation function for our models

def evaluate(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r_square = r2_score(actual, predicted)
    return mae, rmse, r_square

In [11]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list = []
for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)
    
    # Make prediction
    train_prediction = model.predict(X_train)
    test_prediction = model.predict(X_test)
    
    # Evaluate train and test predictions
    train_mae, train_rmse, train_r2 = evaluate(y_train, train_prediction)
    test_mae, test_rmse, test_r2 = evaluate(y_test, test_prediction)
    
    print(name)
    model_list.append(name)

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))
    r2_list.append(test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3370
- Mean Absolute Error: 4.2804
- R2 Score: 0.8737
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3685
- Mean Absolute Error: 4.1820
- R2 Score: 0.8816


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5925
- Mean Absolute Error: 5.2053
- R2 Score: 0.8072
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5173
- Mean Absolute Error: 5.1557
- R2 Score: 0.8254


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3373
- Mean Absolute Error: 4.2790
- R2 Score: 0.8736
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3655
- Mean Absolute Error: 4.1809
- R2 Score: 0.8817


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 5.5979
- Mean Absolute Error: 4.4865
- R2 Score: 0.8610
-----------------------

In [12]:
results_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model_Name', 'R2']).sort_values(by=['R2'], ascending=False)
results_df

,Model_Name,R2
2,Ridge,0.881695
0,Linear Regression,0.881560
8,AdaBoost Regressor,0.855377
7,CatBoosting Regressor,0.852282
5,Random Forest Regressor,0.848300
6,XGBRegressor,0.837903
1,Lasso,0.825446
3,K-Neighbors Regressor,0.783449
4,Decision Tree,0.731875


### Insights
We can see that linear regression or Ridge are the best models for our problem statement